# 📊 03 — Benchmark Report
Runs the full evaluation pipeline on the synthetic test set and shows precision, recall, F1 per entity type.

In [ ]:
import sys, pathlib, subprocess, json
ROOT = pathlib.Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
result = subprocess.run(
    [sys.executable, str(ROOT / 'evaluate.py'), '--out', str(ROOT / 'nb_results.json')],
    capture_output=True, text=True, cwd=str(ROOT)
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

In [ ]:
import pandas as pd
data = json.loads((ROOT / 'nb_results.json').read_text())
rows = []
for etype, m in data.get('by_entity_type', {}).items():
    rows.append({
        'Entity': etype,
        'Precision': round(m['precision'], 3),
        'Recall': round(m['recall'], 3),
        'F1': round(m['f1'], 3),
        'TP': m['tp'], 'FP': m['fp'], 'FN': m['fn'],
    })
df = pd.DataFrame(rows).sort_values('F1', ascending=False)
p, r, f = data['precision'], data['recall'], data['f1']
print(f'Overall — P:{p:.3f}  R:{r:.3f}  F1:{f:.3f}')
print(f'Downstream safe: {data.get("downstream_safe")}')
df